## Experiment summary

For each experiment, count participants who actually recorded that condition, and summarize decoder holdout performance for the participants whose decoders were used.

- **Listed:** IDs in `participant_list`.
- **Had data:** listed IDs with TaskData, LabeledData, and filtered fNIRS for that condition (Passive = `*W`, Active = `*P`). Not every listed participant completed every condition.
- **Used:** participants with a recorded `results_decoder_holdout_<id>` score (passed the holdout-F1 screen and entered the RL runs).

Decoder mean and standard deviation are taken across those used participants (sample standard deviation).

In [22]:
import os
from ast import literal_eval
from pathlib import Path

import pandas as pd

RESULTS_DIR = Path.cwd()
if not any(RESULTS_DIR.glob("*PER.csv")):
    RESULTS_DIR = Path("src/results")

SKIP_PREFIXES = ("best_results", "Z_")


def experiment_files(results_dir: Path) -> list[Path]:
    files = []
    for path in sorted(results_dir.glob("*.csv")):
        if path.name.startswith(SKIP_PREFIXES):
            continue
        header = pd.read_csv(path, nrows=0).columns
        if "results_decoder_holdout_mean" in header:
            files.append(path)
    return files


def holdout_columns(columns) -> list[str]:
    skip = {"results_decoder_holdout_mean", "results_decoder_holdout_std"}
    return [
        c for c in columns
        if c.startswith("results_decoder_holdout_") and c not in skip
    ]


def load_experiment(path: Path) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0).columns.tolist()
    keep = [
        "parameters_experiment_domain",
        "parameters_experiment_task",
        "parameters_experiment_model_granularity",
        "parameters_experiment_participant_list",
        "parameters_experiment_min_holdout_f1",
        "results_decoder_holdout_mean",
        "results_decoder_holdout_std",
    ]
    usecols = [c for c in keep if c in header] + holdout_columns(header)
    df = pd.read_csv(path, usecols=usecols)
    df["source_file"] = path.name
    return df


def condition_codes(domain: str, task: str) -> list[str]:
    letter = str(domain)[0].upper()
    task = str(task).lower()
    if task == "passive":
        return [f"{letter}W"]
    if task == "active":
        return [f"{letter}P"]
    if task == "pooled":
        return [f"{letter}W", f"{letter}P"]
    raise ValueError(f"Unknown task {task!r}")


def participant_data_root() -> Path:
    for key in ("DATA_PATH", "NEUROLOOP_DATA_ROOT"):
        value = os.environ.get(key)
        if value:
            root = Path(value).expanduser()
            if (root / "TaskData").is_dir() and (root / "fNIRS" / "LabeledData").is_dir():
                return root
    fallback = Path("/Users/juliasantaniello/Desktop/fNIRS-2-RL/Experiment/ParticipantData")
    if (fallback / "TaskData").is_dir():
        return fallback
    raise FileNotFoundError("Set DATA_PATH to the ParticipantData root (TaskData + fNIRS/LabeledData).")


def participants_with_condition_data(pids: list[int], codes: list[str], data_root: Path) -> list[int]:
    """Listed participants who have task, label, and filtered fNIRS for every condition code."""
    task_names = os.listdir(data_root / "TaskData")
    label_names = os.listdir(data_root / "fNIRS" / "LabeledData")
    filtered_dirs = [data_root / "fNIRS" / "FilteredData"]
    repo_filtered = Path.cwd().parents[1] / "data" / "fNIRS" / "FilteredData"
    if not repo_filtered.is_dir():
        repo_filtered = Path("/Users/juliasantaniello/Desktop/OfflineNeuroloop/data/fNIRS/FilteredData")
    if repo_filtered.is_dir():
        filtered_dirs.append(repo_filtered)
    filtered_names = []
    for folder in filtered_dirs:
        if folder.is_dir():
            filtered_names.extend(os.listdir(folder))

    kept = []
    for pid in pids:
        tag = f"{int(pid):03d}"
        ok = True
        for code in codes:
            has_task = any(name.startswith(f"{tag}_{code}") for name in task_names)
            has_label = any(name.startswith(f"{tag}_{code}_") for name in label_names)
            has_filtered = any(f"{tag}_processed_{code}" in name for name in filtered_names)
            if not (has_task and has_label and has_filtered):
                ok = False
                break
        if ok:
            kept.append(int(pid))
    return kept


files = experiment_files(RESULTS_DIR)
if not files:
    raise FileNotFoundError(f"No experiment CSVs with decoder scores in {RESULTS_DIR.resolve()}")

DATA_ROOT = participant_data_root()
raw = pd.concat([load_experiment(path) for path in files], ignore_index=True)
print(f"Loaded {len(raw)} rows from {len(files)} file(s)")
print(f"Participant data: {DATA_ROOT}")
for path in files:
    print(f"  {path.name}")

Loaded 2309 rows from 4 file(s)
Participant data: /Users/juliasantaniello/Desktop/fNIRS-2-RL/Experiment/ParticipantData
  RobotPassive_binary_PER.csv
  flappy_finetune_binary_PER.csv
  lunar_passive_binary_PER.csv
  robotPassive_ternary_PER.csv


In [23]:
pid_cols = holdout_columns(raw.columns)

def parse_pids(value) -> list[int]:
    pids = literal_eval(value) if isinstance(value, str) else list(value)
    return [int(pid) for pid in pids]


raw["listed_pids"] = raw["parameters_experiment_participant_list"].map(parse_pids)
raw["n_listed"] = raw["listed_pids"].map(len)
raw["n_used"] = raw[pid_cols].notna().sum(axis=1)

# Data availability depends on domain and task, not on the RL seed.
availability = {}
for (domain, task), group in raw.groupby(
    ["parameters_experiment_domain", "parameters_experiment_task"], sort=False
):
    listed = sorted(set().union(*group["listed_pids"]))
    codes = condition_codes(domain, task)
    with_data = participants_with_condition_data(listed, codes, DATA_ROOT)
    availability[(domain, task)] = {
        "codes": codes,
        "with_data": with_data,
        "n_with_data": len(with_data),
    }

raw["n_with_data"] = [
    availability[(row.parameters_experiment_domain, row.parameters_experiment_task)]["n_with_data"]
    for row in raw.itertuples(index=False)
]
raw["decoder_signature"] = raw[pid_cols].apply(
    lambda row: tuple(row.dropna().round(6)), axis=1
)

# One decoder fit is reused across RL conditions and seeds. Collapse those repeats.
group_keys = [
    "source_file",
    "parameters_experiment_domain",
    "parameters_experiment_task",
    "parameters_experiment_model_granularity",
    "decoder_signature",
]
experiments = (
    raw.groupby(group_keys, dropna=False)
    .agg(
        n_runs=("n_used", "size"),
        n_listed=("n_listed", "first"),
        n_with_data=("n_with_data", "first"),
        n_used=("n_used", "first"),
        decoder_holdout_mean=("results_decoder_holdout_mean", "first"),
        decoder_holdout_std=("results_decoder_holdout_std", "first"),
        min_holdout_f1=("parameters_experiment_min_holdout_f1", "first"),
    )
    .reset_index()
    .drop(columns="decoder_signature")
)

summary = experiments[
    [
        "parameters_experiment_domain",
        "parameters_experiment_task",
        "parameters_experiment_model_granularity",
        "n_runs",
        "n_listed",
        "n_with_data",
        "n_used",
        "decoder_holdout_mean",
        "decoder_holdout_std",
        "min_holdout_f1",
        "source_file",
    ]
].rename(columns={
    "parameters_experiment_domain": "domain",
    "parameters_experiment_task": "task",
    "parameters_experiment_model_granularity": "granularity",
    "n_listed": "participants_listed",
    "n_with_data": "participants_with_data",
    "n_used": "participants_used",
    "decoder_holdout_mean": "mean_decoder_holdout",
    "decoder_holdout_std": "std_decoder_holdout",
})

for (domain, task), info in availability.items():
    print(
        f"{domain} {task} ({', '.join(info['codes'])}): "
        f"{info['n_with_data']} participants had data "
        f"{info['with_data']}"
    )

print(
    f"Across {len(experiments)} experiment(s): "
    f"mean with data = {experiments['n_with_data'].mean():.2f}, "
    f"mean used = {experiments['n_used'].mean():.2f}"
)
summary.round(3)

Robot Passive (RW): 15 participants had data [2, 5, 6, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 24]
Flappy Passive (FW): 14 participants had data [2, 5, 7, 8, 9, 11, 13, 14, 15, 18, 19, 20, 21, 23]
Flappy Active (FP): 8 participants had data [3, 6, 9, 14, 16, 18, 19, 24]
Flappy Pooled (FW, FP): 4 participants had data [9, 14, 18, 19]
Lunar Passive (LW): 10 participants had data [3, 5, 6, 7, 8, 10, 12, 20, 23, 24]
Lunar Active (LP): 12 participants had data [2, 3, 5, 10, 11, 13, 15, 16, 17, 18, 21, 23]
Across 7 experiment(s): mean with data = 11.14, mean used = 2.86


,domain,task,granularity,n_runs,participants_listed,participants_with_data,participants_used,mean_decoder_holdout,std_decoder_holdout,min_holdout_f1,source_file
0,Robot,Passive,binary,329,23,15,5,0.649,0.039,0.55,RobotPassive_binary_PER.csv
1,Flappy,Active,binary,330,23,8,1,0.616,0.000,0.55,flappy_finetune_binary_PER.csv
2,Flappy,Passive,binary,330,23,14,2,0.705,0.055,0.55,flappy_finetune_binary_PER.csv
3,Flappy,Pooled,binary,330,23,4,2,0.747,0.013,0.55,flappy_finetune_binary_PER.csv
4,Lunar,Active,binary,330,23,12,3,0.741,0.189,NaN,lunar_passive_binary_PER.csv
5,Lunar,Passive,binary,330,23,10,3,0.630,0.041,NaN,lunar_passive_binary_PER.csv
6,Robot,Passive,ternary,330,23,15,4,0.462,0.097,0.38,robotPassive_ternary_PER.csv


In [24]:
per_subject = raw.drop_duplicates(group_keys).melt(
    id_vars=[
        "source_file",
        "parameters_experiment_domain",
        "parameters_experiment_task",
        "parameters_experiment_model_granularity",
    ],
    value_vars=pid_cols,
    var_name="participant_column",
    value_name="holdout",
).dropna(subset=["holdout"])

per_subject["participant"] = (
    per_subject["participant_column"]
    .str.removeprefix("results_decoder_holdout_")
)

subject_summary = (
    per_subject.groupby(
        [
            "parameters_experiment_domain",
            "parameters_experiment_task",
            "parameters_experiment_model_granularity",
            "participant",
        ],
        as_index=False,
    )["holdout"]
    .mean()
    .rename(columns={
        "parameters_experiment_domain": "domain",
        "parameters_experiment_task": "task",
        "parameters_experiment_model_granularity": "granularity",
        "holdout": "decoder_holdout",
    })
)

scores = subject_summary["decoder_holdout"]
print(
    f"Used participants: n = {scores.size}\n"
    f"Decoder holdout mean = {scores.mean():.3f}\n"
    f"Decoder holdout std  = {scores.std(ddof=1):.3f}"
)
subject_summary.round(3)

Used participants: n = 20
Decoder holdout mean = 0.636
Decoder holdout std  = 0.136


,domain,task,granularity,participant,decoder_holdout
0,Flappy,Active,binary,009,0.616
1,Flappy,Passive,binary,002,0.760
2,Flappy,Passive,binary,018,0.650
3,Flappy,Pooled,binary,002,0.760
4,Flappy,Pooled,binary,009,0.733
5,Lunar,Active,binary,002,0.665
6,Lunar,Active,binary,005,1.000
7,Lunar,Active,binary,016,0.557
8,Lunar,Passive,binary,005,0.628
9,Lunar,Passive,binary,012,0.581
